# I03 - Batch and Layer Normalization

**Author:** Karthik Arjun  
**LinkedIn:** [karthik-arjun-a5b4a258](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Created:** March 2026  
**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand internal covariate shift problem
2. Implement batch normalization
3. Learn layer normalization and when to use it
4. Understand group and instance normalization
5. Compare different normalization techniques

---

## Prerequisites

- Completed I01-I02
- Understanding of neural network training
- Familiarity with gradient descent

---

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. The Problem: Internal Covariate Shift

### What is Internal Covariate Shift?

**Problem:** During training, the distribution of inputs to each layer changes as the parameters of previous layers change.

**Consequences:**
- Slower training
- Requires careful initialization
- Lower learning rates needed
- Harder to train deep networks

**Solution:** Normalize layer inputs to have consistent distribution

## 2. Batch Normalization

### How Batch Normalization Works

For a mini-batch of activations:

1. **Calculate mean and variance:**
$$\mu_B = \frac{1}{m} \sum_{i=1}^{m} x_i$$
$$\sigma_B^2 = \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu_B)^2$$

2. **Normalize:**
$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

3. **Scale and shift (learnable parameters):**
$$y_i = \gamma \hat{x}_i + \beta$$

**Benefits:**
- Faster training (can use higher learning rates)
- Less sensitive to initialization
- Acts as regularization
- Enables deeper networks

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print(f"Training samples: {x_train.shape}")
print(f"Test samples: {x_test.shape}")

In [ ]:
# Model without batch normalization
def create_model_without_bn():
    model = keras.Sequential([
        layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(2),
        
        layers.Conv2D(64, 3, activation='relu'),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(2),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

# Model with batch normalization
def create_model_with_bn():
    model = keras.Sequential([
        layers.Conv2D(32, 3, input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        
        layers.Conv2D(32, 3),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),
        
        layers.Conv2D(64, 3),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        
        layers.Conv2D(64, 3),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),
        
        layers.Flatten(),
        layers.Dense(128),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        
        layers.Dense(10, activation='softmax')
    ])
    return model

In [ ]:
# Train both models
models = {
    'Without BatchNorm': create_model_without_bn(),
    'With BatchNorm': create_model_with_bn()
}

histories = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history = model.fit(
        x_train, y_train,
        epochs=20,
        batch_size=128,
        validation_split=0.2,
        verbose=0
    )
    
    histories[name] = history.history
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for name, history in histories.items():
    axes[0].plot(history['loss'], label=f'{name} (train)', linewidth=2)
    axes[0].plot(history['val_loss'], label=f'{name} (val)', 
                linestyle='--', linewidth=2)

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for name, history in histories.items():
    axes[1].plot(history['accuracy'], label=f'{name} (train)', linewidth=2)
    axes[1].plot(history['val_accuracy'], label=f'{name} (val)',
                linestyle='--', linewidth=2)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy Comparison', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Layer Normalization

### Difference from Batch Normalization

**Batch Normalization:**
- Normalizes across batch dimension
- Requires batch statistics
- Different behavior in training vs inference

**Layer Normalization:**
- Normalizes across feature dimension
- Independent of batch size
- Same behavior in training and inference

**When to use Layer Norm:**
- RNNs and Transformers
- Small batch sizes
- Online learning
- Reinforcement learning

In [ ]:
# Model with Layer Normalization
def create_model_with_ln():
    model = keras.Sequential([
        layers.Flatten(input_shape=(32, 32, 3)),
        
        layers.Dense(512),
        layers.LayerNormalization(),
        layers.Activation('relu'),
        
        layers.Dense(256),
        layers.LayerNormalization(),
        layers.Activation('relu'),
        
        layers.Dense(128),
        layers.LayerNormalization(),
        layers.Activation('relu'),
        
        layers.Dense(10, activation='softmax')
    ])
    return model

model_ln = create_model_with_ln()
model_ln.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Layer Normalization model created")
model_ln.summary()

## 4. Group Normalization and Instance Normalization

### Group Normalization

**Concept:** Divide channels into groups and normalize within each group

**Benefits:**
- Works well with small batch sizes
- Stable across different batch sizes
- Good for object detection and segmentation

### Instance Normalization

**Concept:** Normalize each channel independently for each sample

**Use cases:**
- Style transfer
- GANs
- Image generation

In [ ]:
# Visualize different normalization techniques
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Create sample data
batch_size, height, width, channels = 4, 8, 8, 16
sample_data = np.random.randn(batch_size, height, width, channels)

# Batch Normalization (normalize across batch)
bn_mean = sample_data.mean(axis=(0, 1, 2), keepdims=True)
bn_std = sample_data.std(axis=(0, 1, 2), keepdims=True)
bn_normalized = (sample_data - bn_mean) / (bn_std + 1e-5)

axes[0, 0].hist(sample_data.flatten(), bins=50, alpha=0.7, label='Original')
axes[0, 0].hist(bn_normalized.flatten(), bins=50, alpha=0.7, label='Batch Norm')
axes[0, 0].set_title('Batch Normalization', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Layer Normalization (normalize across features)
ln_mean = sample_data.mean(axis=(1, 2, 3), keepdims=True)
ln_std = sample_data.std(axis=(1, 2, 3), keepdims=True)
ln_normalized = (sample_data - ln_mean) / (ln_std + 1e-5)

axes[0, 1].hist(sample_data.flatten(), bins=50, alpha=0.7, label='Original')
axes[0, 1].hist(ln_normalized.flatten(), bins=50, alpha=0.7, label='Layer Norm')
axes[0, 1].set_title('Layer Normalization', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Instance Normalization (normalize per sample per channel)
in_mean = sample_data.mean(axis=(1, 2), keepdims=True)
in_std = sample_data.std(axis=(1, 2), keepdims=True)
in_normalized = (sample_data - in_mean) / (in_std + 1e-5)

axes[1, 0].hist(sample_data.flatten(), bins=50, alpha=0.7, label='Original')
axes[1, 0].hist(in_normalized.flatten(), bins=50, alpha=0.7, label='Instance Norm')
axes[1, 0].set_title('Instance Normalization', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Comparison
axes[1, 1].hist(bn_normalized.flatten(), bins=50, alpha=0.5, label='Batch Norm')
axes[1, 1].hist(ln_normalized.flatten(), bins=50, alpha=0.5, label='Layer Norm')
axes[1, 1].hist(in_normalized.flatten(), bins=50, alpha=0.5, label='Instance Norm')
axes[1, 1].set_title('All Normalizations Compared', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. When to Use Each Normalization

### Decision Guide

| Normalization | Best For | Batch Size | Use Case |
|--------------|----------|------------|----------|
| **Batch Norm** | CNNs | Large (>32) | Image classification, object detection |
| **Layer Norm** | RNNs, Transformers | Any | NLP, sequence models |
| **Group Norm** | CNNs | Small (<16) | Object detection, segmentation |
| **Instance Norm** | Style transfer | Any | GANs, image generation |

### Practical Tips

1. **Start with Batch Norm** for CNNs with large batches
2. **Use Layer Norm** for Transformers and RNNs
3. **Try Group Norm** if batch size is small
4. **Place normalization** before or after activation (experiment)
5. **Don't normalize** the output layer

In [ ]:
# Complete example: ResNet-style block with Batch Normalization
def residual_block(x, filters, kernel_size=3):
    """ResNet-style residual block with Batch Normalization"""
    # Save input for skip connection
    shortcut = x
    
    # First conv block
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Second conv block
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Add skip connection
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

# Build model with residual blocks
def create_resnet_model():
    inputs = keras.Input(shape=(32, 32, 3))
    
    x = layers.Conv2D(32, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Residual blocks
    x = residual_block(x, 32)
    x = residual_block(x, 32)
    
    x = layers.MaxPooling2D(2)(x)
    
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    outputs = layers.Dense(10, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model

resnet_model = create_resnet_model()
print("ResNet-style model with Batch Normalization created")
resnet_model.summary()

## Summary

In this lesson, you learned:

1. ✅ Internal covariate shift and why normalization helps
2. ✅ Batch normalization theory and implementation
3. ✅ Layer normalization for RNNs and Transformers
4. ✅ Group and instance normalization
5. ✅ When to use each normalization technique

**Key Takeaways:**
- Normalization accelerates training and improves stability
- Batch Norm is default for CNNs with large batches
- Layer Norm is better for RNNs and Transformers
- Choose normalization based on architecture and batch size
- Normalization acts as regularization

**Next Steps:**
- Move to I04 - Advanced CNN Architectures
- Experiment with different normalization techniques
- Try combining normalization with regularization

---

**References:**
- Ioffe & Szegedy (2015): "Batch Normalization: Accelerating Deep Network Training"
- Ba et al. (2016): "Layer Normalization"
- Wu & He (2018): "Group Normalization"
- Ulyanov et al. (2016): "Instance Normalization"